### Online interactive supplemental material for Sandle et al., 2026

These are the corresponding online supplemental materials for our review of latent variable models. This site contains some interactive figures. If you would like to investigate our full database of studies more closely, you can launch our [study explorer](https://studyexplorer-latentvariables-externalizing.streamlit.app/). If you would like to go through an interactive version of our decision tree to help choose a model for your analysis, you can click [here](https://decisiontree-latentvariables-externalizing.streamlit.app/).

If you have any questions, please contact us at zoe.sandle@donders.ru.nl

In [1]:
import sys
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import os
from pathlib import Path
work_dir = Path(r'C:/Users/U727148/OneDrive - Radboud Universiteit/DATA/REVIEW/final_analysis')
os.chdir(work_dir)
sys.path.append('C:/Users/U727148/OneDrive - Radboud Universiteit/DATA/REVIEW/final_analysis')


In [ ]:
dfsankey_models = pd.read_excel('dfsankey_models.xlsx')
#update count variables for this replace gm_count with the count of unique model specifications for each gm

In [3]:
dfsankey_models['gm'].value_counts()
dfsankey_models['gm_count'] = dfsankey_models['gm'].map(dfsankey_models['gm'].value_counts())
dfsankey_models['oa_count'] = dfsankey_models['other_analysis'].map(dfsankey_models['other_analysis'].value_counts())
dfsankey_models['mit_count'] = dfsankey_models['mit'].map(dfsankey_models['mit'].value_counts())

In [5]:

# 1. Split into the two "hops" of the Sankey
hop1 = dfsankey_models.groupby(['gm', 'mit']).agg({
    'gm_count': 'sum', #count of source (first)
    'nameyear': lambda x: '<br>'.join(sorted(x.unique()))
}).reset_index().rename(columns={'gm': 'source', 'mit': 'target', 'gm_count': 'value'})

hop2 = dfsankey_models.groupby(['mit', 'other_analysis']).agg({
    'oa_count': 'sum', #count of target (second)
    'nameyear': lambda x: '<br>'.join(sorted(x.unique()))
}).reset_index().rename(columns={'mit': 'source', 'other_analysis': 'target', 'oa_count': 'value'})

# 2. Combine hops into one flow table
df_links = pd.concat([hop1, hop2], axis=0)

# 3. Create the unique labels for the nodes
labels = list(pd.concat([df_links['source'], df_links['target']]).unique())
label_map = {label: idx for idx, label in enumerate(labels)}

# 4. Map labels to indices
sources = df_links['source'].map(label_map).tolist()
targets = df_links['target'].map(label_map).tolist()
values = df_links['value'].tolist()
customdata = df_links['nameyear'].tolist()

# Create the Sankey plot
#make the flows colors based on the source node
# Build link colors from each link's source node
#unique_sources = pd.unique(df_links["source"])
#palette = sns.color_palette("Pastel2", n_colors=len(unique_sources)).as_hex()
#source_color_map = dict(zip(unique_sources, palette))
#link_colors = df_links["source"].map(source_color_map).tolist()

import seaborn as sns
node_colors = sns.color_palette("pastel", len(labels)).as_hex()
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15, 
        thickness=20,
        line=dict(color="black", width=0.9),
        label=labels,
        color=node_colors,
        align="left"
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        customdata=customdata,
        #color=link_colors,
        hovertemplate='<b>Studies in this flow:</b><br>%{customdata}<extra></extra>',
        hovercolor='rgba(0, 0, 0, 0.5)'
    )
)])
# make the title font size larger than the other text in the figure, and make the figure larger
# and offset title from the top of the figure, and make the hover text larger

fig.update_layout(title_text=f"Sankey Diagram: General Model, Model Specifications, Follow-Up analyses", title_font_size=30, font_size=15, width=1400, height=3000, hovermode='x')
fig.show()
#save the figure as a html file
save_path = r"C:\Users\U727148\Desktop\DATA\REVIEW\sankey_plots_LV_review\sankey_diagram_models.html"
fig.write_html(save_path)


In [7]:
dfsankeybeh = pd.read_excel('dfsankey_beh.xlsx')

In [8]:

# 1. Split into the two "hops" of the Sankey
hop1 = dfsankeybeh.groupby(['behavioral_variable', 'behavioral_assessment']).agg({
    'b_count': 'sum', 
    'nameyear': lambda x: '<br>'.join(sorted(x.unique()))
}).reset_index().rename(columns={'behavioral_variable': 'source', 'behavioral_assessment': 'target', 'b_count': 'value'})

hop2 = dfsankeybeh.groupby(['behavioral_assessment', 'behavioral_assessment_type']).agg({
    'behavioral_assessment_count': 'sum', 
    'nameyear': lambda x: '<br>'.join(sorted(x.unique()))
}).reset_index().rename(columns={'behavioral_assessment': 'source', 'behavioral_assessment_type': 'target', 'behavioral_assessment_count': 'value'})

# 2. Combine hops into one flow table
df_links = pd.concat([hop1, hop2], axis=0)

# 3. Create the unique labels for the nodes
labels = list(pd.concat([df_links['source'], df_links['target']]).unique())
label_map = {label: idx for idx, label in enumerate(labels)}

# 4. Map labels to indices
sources = df_links['source'].map(label_map).tolist()
targets = df_links['target'].map(label_map).tolist()
values = df_links['value'].tolist()
customdata = df_links['nameyear'].tolist()

import seaborn as sns
node_colors = sns.color_palette("pastel", len(labels)).as_hex()
# Create the Sankey plot
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15, 
        thickness=20,
        line=dict(color="black", width=0.9),
        label=labels,
        color=node_colors,
        align="left"
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        customdata=customdata,
        # Using %{customdata} now displays the unique, newline-separated list
        hovertemplate='<b>Studies in this flow:</b><br>%{customdata}<extra></extra>',
        hovercolor='rgba(0, 0, 0, 0.5)'
    ))])

fig.update_layout(title_text="Sankey Diagram: Behavioral Variable, Assessment, Assessemt Type", font_size=13.5, width=1700, height=2000, hovermode='x')
fig.show()
save_path = r"C:\Users\U727148\Desktop\DATA\REVIEW\sankey_plots_LV_review\sankey_diagram_behavioral.html"
fig.write_html(save_path)

 

In [16]:
dfsankeycu = pd.read_excel('dfsankey_cu.xlsx')

In [17]:
dfsankeycog = pd.read_excel('dfsankey_cog1counts1.xlsx')

In [23]:

# 1. Split into the two "hops" of the Sankey
hop1 = dfsankeycog.groupby(['cognitive_variable', 'cognitive_assessment_type']).agg({
    'cognitive_variable_count': 'sum', 
    'nameyear': lambda x: '<br>'.join(sorted(x.unique()))
}).reset_index().rename(columns={'cognitive_variable': 'source', 'cognitive_assessment_type': 'target', 'cognitive_variable_count': 'value'})

hop2 = dfsankeycog.groupby(['cognitive_assessment_type', 'Cognitive Assessment EXP sum']).agg({
    'cognitive_assessment_type_count': 'sum', 
    'nameyear': lambda x: '<br>'.join(sorted(x.unique()))
}).reset_index().rename(columns={'cognitive_assessment_type': 'source', 'Cognitive Assessment EXP sum': 'target', 'cognitive_assessment_type_count': 'value'})

hop3 = dfsankeycog.groupby(['Cognitive Assessment EXP sum', 'cog_asses_experimanttasks']).agg({
    'cog_assessment_count': 'sum', 
    'nameyear': lambda x: '<br>'.join(sorted(x.unique()))
}).reset_index().rename(columns={'Cognitive Assessment EXP sum': 'source', 'cog_asses_experimanttasks': 'target', 'cog_assessment_count': 'value'})

# 2. Combine hops into one flow table
df_links = pd.concat([hop1, hop2, hop3], axis=0)

# 3. Create the unique labels for the nodes
labels = list(pd.concat([df_links['source'], df_links['target']]).unique())
label_map = {label: idx for idx, label in enumerate(labels)}

# 4. Map labels to indices
sources = df_links['source'].map(label_map).tolist()
targets = df_links['target'].map(label_map).tolist()
values = df_links['value'].tolist()
customdata = df_links['nameyear'].tolist()
# add muted colorpalette for the nodes
import seaborn as sns
node_colors = sns.color_palette("pastel", len(labels)).as_hex()
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15, 
        thickness=20,
        line=dict(color="black", width=0.9),
        color=node_colors,
        label=labels,
        align="left"
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        customdata=customdata,
        # Using %{customdata} now displays the unique, newline-separated list
        hovertemplate='<b>Studies in this flow:</b><br>%{customdata}<extra></extra>',
        hovercolor='rgba(0, 0, 0, 0.5)'
    ))])

fig.update_layout(title_text="Sankey Diagram: Cognitive Variables, Assessments, and experimental tasks", font_size=12.5, width=1700, height=2000, hovermode='x')
fig.show()
save_path = r"C:\Users\U727148\Desktop\DATA\REVIEW\sankey_plots_LV_review\sankey_diagram_cognitive.html"
fig.write_html(save_path)


In [24]:

# 1. Split into the two "hops" of the Sankey
hop1 = dfsankeycu.groupby(['p_or_cu', 'scale']).agg({
    'pcu_count': 'sum', 
    'nameyear': lambda x: '<br>'.join(sorted(x.unique()))
}).reset_index().rename(columns={'p_or_cu': 'source', 'scale': 'target', 'pcu_count': 'value'})

hop2 = dfsankeycu.groupby(['scale', 'reporting_type']).agg({
    'reporttype_count': 'sum', 
    'nameyear': lambda x: '<br>'.join(sorted(x.unique()))
}).reset_index().rename(columns={'scale': 'source', 'reporting_type': 'target', 'reporttype_count': 'value'})

# 2. Combine hops into one flow table
df_links = pd.concat([hop1, hop2], axis=0)

# 3. Create the unique labels for the nodes
labels = list(pd.concat([df_links['source'], df_links['target']]).unique())
label_map = {label: idx for idx, label in enumerate(labels)}

# 4. Map labels to indices
sources = df_links['source'].map(label_map).tolist()
targets = df_links['target'].map(label_map).tolist()
values = df_links['value'].tolist()
customdata = df_links['nameyear'].tolist()

import seaborn as sns
node_colors = sns.color_palette("pastel", len(labels)).as_hex()
# Create the Sankey plot
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15, 
        thickness=20,
        line=dict(color="black", width=0.9),
        color=node_colors,
        label=labels,
        align="left"
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        customdata=customdata,
        # Using %{customdata} now displays the unique, newline-separated list
        hovertemplate='<b>Studies in this flow:</b><br>%{customdata}<extra></extra>',
        hovercolor='rgba(0, 0, 0, 0.5)'
    ))])

fig.update_layout(title_text="Sankey Diagram: Psychopathic vs CU traits, Scale, Reporting", font_size=16.5, width=1500, height=1800, hovermode='x')
fig.show()
save_path = r"C:\Users\U727148\Desktop\DATA\REVIEW\sankey_plots_LV_review\sankey_diagram_psycho.html"
fig.write_html(save_path)


In [20]:
dfsankey_brain = pd.read_excel('dfsankey_brain_exploded.xlsx')

In [25]:
# Create the unique labels for the nodes
labels = list(pd.concat([dfsankey_brain['brain variable'], dfsankey_brain['brain areas of interest']]).unique())

# Map labels to indices
label_map = {label: idx for idx, label in enumerate(labels)}

# Define the sources, targets, and values based on the new DataFrame
sources = (
    dfsankey_brain['brain variable'].map(label_map).tolist() 
    )
targets = (
    dfsankey_brain['brain areas of interest'].map(label_map).tolist()
)
values = (
    dfsankey_brain['brain_variable_count'].tolist() +
    dfsankey_brain['brain_areas_of_interest_count'].tolist()
)


import seaborn as sns
node_colors = sns.color_palette("pastel", len(labels)).as_hex()
# Create the Sankey plot using Plotly
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.9),
        label=labels,
        color=node_colors,
        align="left"
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        customdata=dfsankey_brain['nameyear'].tolist() + dfsankey_brain['nameyear'].tolist(),
        hovertemplate='%{customdata}<extra></extra>',
        hovercolor='rgba(0, 0, 0, 0.5)'
    ))])

fig.update_layout(title_text="Overview Brain methods and ROIs", font_size=14.5, width=1200, height=1000, hovermode='x')
fig.show()
save_path = r"C:\Users\U727148\Desktop\DATA\REVIEW\sankey_plots_LV_review\sankey_diagram_brain.html"
fig.write_html(save_path)


The following sankey plots show the flow between different aspects of model use and variable Measurement. You can hover over the flows to see the studies in it. 

Here you can see model use. On the left is the general model as used in the text of our review. In the middle you can see more specific descriptions and model specifications. On the right you can see the (not necessarily latent) follow up analyses.

In [37]:
#embed the html files in the notebook using an iframe
file_path = r"C:\Users\U727148\Desktop\DATA\REVIEW\sankey_plots_LV_review"

IFrame("sankey_diagram_models.html",width=1400, height=3000)

Next are the behavioral (EXT) variables. Left is the variable in question, middle is the scale/tool used, and on the right is assessment modality.

In [38]:
IFrame("sankey_diagram_behavioral.html", width=1700, height=2000)

In the cognitive variable plot, the cognitive variable is on the left, the next hop shows the assessment modality, and the next two hops show summarized and specific tool used.

In [39]:
IFrame("sankey_diagram_cognitive.html", width=1700, height=2000)

In the plot showing psychopathic/CU traits, the variable is once again on the left, the scale in the middle, and the assessment modality is on the right.

In [40]:
IFrame("sankey_diagram_psycho.html", width=1700, height=2000)

Lastly, on the left we have the assessment modality for the brain and on the right we see the brain region(s).

In [41]:
IFrame("sankey_diagram_brain.html", width=1400, height=1000)

In [42]:
import pandas as pd
import networkx as nx
from pyvis.network import Network
from pathlib import Path
review_path = Path("C:\\Users\\U727148\\OneDrive - Radboud Universiteit\\DATA\\REVIEW")
import os
os.chdir(review_path)

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_excel("network_LA_review.xlsx")

# ============================================================
# NODE DEFINITIONS
# ============================================================

node_mapping = {
    "behavior_predictor": ("Behavior Predictor", "Predictor"),
    "behavior_outcome": ("Behavior Outcome", "Outcome"),

    "cognition_predictor": ("Cognition Predictor", "Predictor"),
    "cognition_outcome": ("Cognition Outcome", "Outcome"),

    "brain_predictor": ("Brain Predictor", "Predictor"),
    "brain_outcome": ("Brain Outcome", "Outcome"),

    "p/cu_predictor": ("P/CU Predictor", "Predictor"),
    "p/cu_outcome": ("P/CU Outcome", "Outcome"),

    "behavior_mod": ("Behavior Moderator", "Moderator"),
    "cog_mod": ("Cognition Moderator", "Moderator"),
    "p/cu_mod": ("P/CU Moderator", "Moderator"),
}

# ============================================================
# COLOR MAP
# ============================================================

node_color = {
    "behavior_predictor": 'rgb(255, 107, 107)',
    "behavior_outcome": 'rgb(255, 107, 107)',
    "behavior_mod": 'rgb(255, 107, 107)',

    "cognition_predictor": 'rgb(107, 255, 107)',
    "cognition_outcome": 'rgb(107, 255, 107)',
    "cog_mod": 'rgb(107, 255, 107)',

    "brain_predictor": 'rgb(107, 107, 255)',
    "brain_outcome": 'rgb(107, 107, 255)',

    "p/cu_predictor": 'rgb(255, 255, 107)',
    "p/cu_outcome": 'rgb(255, 255, 107)',
    "p/cu_mod": 'rgb(255, 255, 107)',
}



# ============================================================
# NODE COUNTS
# ============================================================

node_counts = {}

for col in df.columns:
    node_name = node_mapping[col][0]
    node_counts[node_name] = int(df[col].sum())

# ============================================================
# EDGE COUNTS
# ============================================================

edge_counts = {}

predictors = [c for c in df.columns if "predictor" in c]
outcomes = [c for c in df.columns if "outcome" in c]
moderators = [c for c in df.columns if "mod" in c]

for _, row in df.iterrows():

    active_preds = [p for p in predictors if row[p] == 1]
    active_outs = [o for o in outcomes if row[o] == 1]
    active_mods = [m for m in moderators if row[m] == 1]

    # predictor → outcome
    for p in active_preds:
        for o in active_outs:
            edge = (node_mapping[p][0], node_mapping[o][0])
            edge_counts[edge] = edge_counts.get(edge, 0) + 1

    # moderator → predictor/outcome
    for m in active_mods:
        for p in active_preds:
            edge = (node_mapping[m][0], node_mapping[p][0])
            edge_counts[edge] = edge_counts.get(edge, 0) + 1

        for o in active_outs:
            edge = (node_mapping[m][0], node_mapping[o][0])
            edge_counts[edge] = edge_counts.get(edge, 0) + 1

# ============================================================
# BUILD NETWORK GRAPH
# (kept for consistency, not required for PyVis)
# ============================================================

G = nx.DiGraph()

for node, count in node_counts.items():
    G.add_node(node, count=count)

for (source, target), count in edge_counts.items():
    G.add_edge(source, target, weight=count)

# ============================================================
# PYVIS INTERACTIVE NETWORK (THIS REPLACES PLOTLY)
# ============================================================

net = Network(
    height="900px",
    width="100%",
    bgcolor="white",
    directed=True,
    notebook=False
)

# physics = draggable + force-directed layout
net.barnes_hut()

# ============================================================
# ADD NODES
# ============================================================
always_label = {
    "Behavior Predictor",
    "Behavior Outcome",
    "Cognition Predictor",
    "Cognition Outcome",
    "Brain Predictor",
    "Brain Outcome",
    "P/CU Predictor",
    "P/CU Outcome"
    'Behavior Moderator',
    'Cognition Moderator',
    'P/CU Moderator'
}
for node in G.nodes():

    raw_col = [k for k, (name, _) in node_mapping.items() if name == node][0]
    color = node_color.get(raw_col, "gray")

    net.add_node(
        node,
        label=node if node in always_label else "",
        size=20 + node_counts[node] * 3,
        color=color,
        title=f"{node}\nUses: {node_counts[node]}",
        font={"size":80} if node in always_label else {"size": 0}
    )

# ============================================================
# ADD EDGES
# ============================================================

for (source, target), count in edge_counts.items():

    net.add_edge(
        source,
        target,
        value=count,
        size=edge_counts[edge],
        color='rgba(0,0,0)',
        title=f"{source} → {target}: {count}"
    )


# ============================================================
# RENDER OUTPUT
# ============================================================

html = net.generate_html()

legend_html = """
<div style="
    width:100%;
    display:flex;
    justify-content:center;
    gap:30px;
    margin-bottom:15px;
    font-family:Arial;
">

    <div><span style="
        display:inline-block;
        width:18px;
        height:18px;
        background:rgb(255, 107, 107);
        margin-right:8px;
    "></span>Behavior</div>

    <div><span style="
        display:inline-block;
        width:18px;
        height:18px;
        background:rgb(107, 255, 107);
        margin-right:8px;
    "></span>Cognition</div>

    <div><span style="
        display:inline-block;
        width:18px;
        height:18px;
        background:rgb(107, 107, 255);
        margin-right:8px;
    "></span>Brain</div>

    <div><span style="
        display:inline-block;
        width:18px;
        height:18px;
        background:rgb(255, 255, 107);
        margin-right:8px;
    "></span>Psychopathic & CU traits</div>
</div>
"""

title_html = """
<h2 style="
    text-align:center;
    font-family:Arial;
    margin-top:20px;
">
    Network of Predictors, Outcomes, and Moderators
    in Externalizing Behavior Research
    (Sandle et al., 2026)
</h2>
"""

# insert directly after <body>
html = html.replace(
    "<body>",
    f"<body>{title_html}{legend_html}"
)

with open("pred_out_mod_network.html", "w", encoding="utf-8") as f:
    f.write(html)

from IPython.display import IFrame
import urllib.parse

src = "data:text/html;charset=utf-8," + urllib.parse.quote(html)

IFrame(src, width="100%", height="900px")
#save as a png with 300dpi
#net.save_graph("pred_out_mod_network.html")